# Encoding Technique 5: Target Encoding

**Dataset:** `Loan_Default.csv`

**When to use:** For **nominal** features with **high cardinality**, especially in regression/classification tasks where you want to capture the relationship between a category and the target variable.

**Key concept:** Each category is replaced by the **mean of the target variable** for that category. For example, if the `loan_amount` mean for `'North'` region is `285,000`, then every `'North'` entry in the `Region` column is replaced by `285000`.

---


### Step 1: Setup, Data Loading & Prep


In [1]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
import category_encoders as ce

# Load data
df = pd.read_csv('../../data/raw/Loan_Default.csv')
df.drop(['ID', 'year'], axis=1, inplace=True)

categorical_features = df.select_dtypes(include=['object']).columns.tolist()
Ordinal_features = ['age']
Nominal_features = categorical_features.copy()
Nominal_features.remove('age')

# Encode the ordinal feature first
enc = OrdinalEncoder()
df[Ordinal_features] = enc.fit_transform(df[Ordinal_features])

print(f'Shape before Target Encoding: {df.shape}')
df.head()

Shape before Target Encoding: (148670, 32)


,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,loan_amount,rate_of_interest,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,116500,NaN,...,EXP,758,CIB,0.0,to_inst,98.728814,south,direct,1,45.0
1,cf,Male,nopre,type2,p1,l1,nopc,b/c,206500,NaN,...,EQUI,552,EXP,3.0,to_inst,NaN,North,direct,1,NaN
2,cf,Male,pre,type1,p1,l1,nopc,nob/c,406500,4.56,...,EXP,834,CIB,1.0,to_inst,80.019685,south,direct,0,46.0
3,cf,Male,nopre,type1,p4,l1,nopc,nob/c,456500,4.25,...,EXP,587,CIB,2.0,not_inst,69.376900,North,direct,0,42.0
4,cf,Joint,pre,type1,p1,l1,nopc,nob/c,696500,4.00,...,CRIF,602,EXP,0.0,not_inst,91.886544,North,direct,0,39.0


### Step 2: Apply Target Encoding

The target variable used is `loan_amount`. Notice that `fit_transform` takes **both X (features) and y (target)** — this is what makes it a _supervised_ encoding method.


In [4]:
df_target = df.copy()

# Configure the Target Encoder
Target_encoder = ce.TargetEncoder(cols=Nominal_features)

# Separate numerical and categorical features
# The ordinal is now numerical, it's been encoded dude
df_target_numerical = df_target.drop(Nominal_features, axis=1)

# fit_transform requires the TARGET variable (y)
df_target_categorical = Target_encoder.fit_transform(
    df_target[Nominal_features],
    df_target['loan_amount']   # <-- the target variable
)

# Reassemble
df_target = pd.concat([df_target_numerical, df_target_categorical], axis=1)

print(f'Shape after Target Encoding: {df_target.shape}')
df_target['Gender'].head()

Shape after Target Encoding: (148670, 32)


0    293126.304469
1    331420.889812
2    331420.889812
3    331420.889812
4    388587.248484
Name: Gender, dtype: float64

### Step 3: Inspect the Encoded Values

The nominal columns now contain **continuous mean values** of `loan_amount` per category, instead of string labels.


In [3]:
# Compare: original Gender categories → encoded mean loan_amount values
print('Target-encoded Gender unique values:')
print(df_target['Gender'].unique())

Target-encoded Gender unique values:
[293126.30446905 331420.8898125  388587.24848426 295861.4758307 ]


### Key Observation

Each category is now a **mean of the target**. This approach is extremely compact but requires careful handling (cross-validation) to prevent data leakage in production models.
